# QuantumSignals x Interactive Brokers — Trading Tutorial

This notebook walks you through every step of the live-trading pipeline end-to-end, from connecting to Interactive Brokers all the way to following QuantumSignals and placing real (or simulated) orders.

Run each cell in order. By default the bot uses **dry-run mode** (simulated orders). Set `DRY_RUN=false` in your `.env` file to route orders through IB Gateway.

## Prerequisites

### 1. IB Gateway (or TWS) running on localhost

- Download at https://www.interactivebrokers.com/en/trading/ibgateway.php
- Log in with your IB paper-trading or live account credentials.
- In **Settings → API → Settings**:
  - ✅ Enable ActiveX and Socket Clients
  - ✅ Socket port: `4002` (paper) / `4001` (live)
  - ✅ Allow connections from localhost (`127.0.0.1`)

### 2. QuantumSignals API key

Stored in a `.env` file at the project root:

```
QUANTUMSIGNALS_API_KEY=your_key_here
```

### 3. Optional `.env` overrides

| Variable | Default | Description |
|---|---|---|
| `DRY_RUN` | `true` | `false` → real IB orders |
| `IB_HOST` | `127.0.0.1` | IB Gateway host |
| `IB_PORT` | `4002` | `4002` = paper gateway, `7497` = TWS paper |
| `IB_CLIENT_ID` | `1` | IB API client ID |
| `IB_MARKET_DATA_TYPE` | `4` | `1`=live, `3`=delayed, `4`=delayed-frozen (free) |
| `ALLOWED_SYMBOLS` | *(all)* | Comma-separated, e.g. `ES,NQ` |
| `EXECUTION_INTERVAL_MINUTES` | `5` | Trading frequency in minutes |

---

## Imports & Logging Setup

In [1]:
from __future__ import annotations

import os
import queue
import sys
import threading
import time
from pathlib import Path

from dotenv import load_dotenv
from loguru import logger
from quantumsignals.client import Client as QSClient, APIError, APIKeyError

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "trader").is_dir():
        _SRC = str(_root / "src")
        if _SRC not in sys.path:
            sys.path.insert(0, _SRC)
        break

from trader.config import Settings, load_settings
from trader.gateways import create_gateway
from trader.models import PerformanceTracker, Signal

In [2]:
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level:<8}</level> | {message}",
    level="DEBUG",
    colorize=True,
)

1

---

## Step 1 — Load Configuration

All settings come from environment variables (loaded from `.env` by `load_dotenv`) and are parsed into a frozen `Settings` dataclass.

Key variables:
- **`QUANTUMSIGNALS_API_KEY`** — required
- **`DRY_RUN`** — `true` uses `SimulatedGateway` (no real orders), `false` uses `IBGateway`
- **`STRATEGY`** — `"momentum"` or `"signal_follower"`
- **`ALLOWED_SYMBOLS`** — comma-separated list, e.g. `"ES,NQ"`
- **`CLOSE_ON_EXIT`** — `true` flattens all positions on shutdown
- **`DAILY_START_CAPITAL`** — used for return-% calculation (default `100000`)

In [3]:
load_dotenv()

settings = load_settings()

logger.info(f"Dry-run mode  : {'YES — orders go to SimulatedGateway' if settings.dry_run else 'NO  — orders go to IB Gateway'}")
logger.info(f"IB Gateway    : {settings.ib_host}:{settings.ib_port}  clientId={settings.ib_client_id}")
logger.info(f"Market data   : type {settings.ib_market_data_type} (4=delayed-frozen, 1=live)")
logger.info(f"Strategy      : {settings.strategy}")
logger.info(f"Exec interval : {settings.execution_interval_minutes} minutes")
logger.info(f"Allowed syms  : {', '.join(settings.allowed_symbols) or '(all)'}")

19:24:00 | INFO     | Dry-run mode  : NO  — orders go to IB Gateway
19:24:00 | INFO     | IB Gateway    : 127.0.0.1:4002  clientId=1
19:24:00 | INFO     | Market data   : type 3 (4=delayed-frozen, 1=live)
19:24:00 | INFO     | Strategy      : momentum
19:24:00 | INFO     | Exec interval : 5 minutes
19:24:00 | INFO     | Allowed syms  : ES, NQ


---

## Step 2 — Connect to Interactive Brokers

`create_gateway()` returns an **`IBGateway`** (when `DRY_RUN=false`) or a **`SimulatedGateway`** (when `DRY_RUN=true`).

The `.connect()` call does three things:
1. Starts `ib_async`'s asyncio event loop on this thread.
2. Opens a TCP connection to IB Gateway / TWS.
3. Sets the requested market-data tier.

> **Threading rule:** `ib_async` ties its event loop to the thread that called `connect()`. Never call IB API methods from background threads — they will deadlock or return empty results.

In [ ]:
gw = create_gateway(settings)
logger.info(f"Gateway type: {type(gw).__name__}")

try:
    gw.connect()
    logger.success("IB Gateway connection successful")
except Exception as exc:
    logger.critical(f"Cannot connect to IB Gateway: {exc}")
    logger.critical("Make sure IB Gateway or TWS is running and API connections are enabled.")
    raise

19:25:01 | INFO     | Gateway type: IBGateway
19:25:01 | DEBUG    | IB notice 2104: Market data farm connection is OK:usfarm
19:25:01 | DEBUG    | IB notice 2106: HMDS data farm connection is OK:ushmds
19:25:01 | DEBUG    | IB notice 2158: Sec-def data farm connection is OK:secdefil
19:25:01 | SUCCESS  | Connected to IB Gateway  127.0.0.1:4002  (clientId=1, marketDataType=delayed)
19:25:01 | SUCCESS  | IB Gateway connection successful


19:30:55 | DEBUG    | IB notice 2107: HMDS data farm connection is inactive but should be available upon demand.ushmds


Quick account snapshot to confirm the connection is live:

In [6]:
account = gw.get_account_summary()

logger.info(f"Net liquidation : ${account['net_liquidation']:>12,.2f}")
logger.info(f"Buying power    : ${account['buying_power']:>12,.2f}")
logger.info(f"Unrealised PnL  : ${account['unrealized_pnl']:>12,.2f}")

19:25:04 | INFO     | Net liquidation : $  746,384.97
19:25:04 | INFO     | Buying power    : $  482,033.64
19:25:04 | INFO     | Unrealised PnL  : $        0.00


---

## Step 3 — Connect to QuantumSignals & Browse the Model Catalog

`QSClient` is a context-manager HTTP + SSE client. `get_model_catalog()` returns a dict of `{model_id: ModelInfo}` where each `ModelInfo` has:

| Attribute | Type | Example |
|---|---|---|
| `trained_on_symbols` | `list[str]` | `["ES", "NQ"]` |
| `trained_date_range` | `(str, str)` | ISO date strings |

This is purely informational — it tells you which symbols each model was trained on.

In [7]:
try:
    with QSClient(api_key=settings.qs_api_key) as qs:
        logger.success("QuantumSignals API connection successful")

        models = qs.get_model_catalog()
        logger.info(f"Available models: {len(models)}")

        for model_id, model in models.items():
            symbols = ", ".join(model.trained_on_symbols)
            date_range = f"{model.trained_date_range[0]} → {model.trained_date_range[1]}"
            logger.info(f"  Model {model_id:>4s} | symbols: {symbols:<18s} | trained: {date_range}")

except APIKeyError as exc:
    logger.critical(f"Invalid API key: {exc}")
    raise
except Exception as exc:
    logger.warning(f"Could not fetch model catalog: {exc}")

2026-05-06 19:25:07 [info     ] client_initialized             base_url=https://api.quantumsignals.ai/q version=0.4.0


19:25:07 | SUCCESS  | QuantumSignals API connection successful
19:25:10 | INFO     | Available models: 3
19:25:10 | INFO     |   Model Pythia-gC3PO | symbols: YM                 | trained: 2024-09-30 → 2025-09-30
19:25:10 | INFO     |   Model Pythia-Br2d2 | symbols: RTY                | trained: 2024-09-30 → 2025-09-30
19:25:10 | INFO     |   Model Pythia-aB24X | symbols: ES, NQ             | trained: 2024-09-30 → 2025-09-30


---

## Step 4 — Stream QuantumSignals Signals (Background Thread)

`qs.stream_signals()` is a blocking SSE (Server-Sent Events) iterator. Each yielded signal object has:

| Attribute | Type | Description |
|---|---|---|
| `symbol` | `str` | e.g. `"ES"`, `"NQ"` |
| `signal` | `int` | `0`=DOWN, `1`=STABLE, `2`=UP (matches `Signal` enum) |
| `model` | `str` | Which model produced this signal |
| `for_minute_utc` | `datetime` | The 1-minute bar this signal is for |
| `is_null_signal()` | `bool` | `True` for keepalive / out-of-hours heartbeats |

Since `ib_async` owns the main thread, we push signals onto a thread-safe `queue.Queue` and consume them on the main thread.

**Retry logic:** `APIKeyError` stops immediately; other errors use exponential back-off up to `max_stream_retries`.

In [8]:
def _signal_producer(
    settings: Settings,
    stop_event: threading.Event,
    signal_queue: "queue.Queue[object | None]",
) -> None:
    """Background thread: streams QS signals into signal_queue."""
    retries = 0
    max_retries = settings.max_stream_retries

    try:
        while not stop_event.is_set() and retries < max_retries:
            try:
                with QSClient(api_key=settings.qs_api_key) as qs:
                    logger.success("QuantumSignals SSE stream connected")
                    retries = 0

                    for signal in qs.stream_signals():
                        if stop_event.is_set():
                            break
                        signal_queue.put(signal)

            except APIKeyError as exc:
                logger.critical(f"Invalid QuantumSignals API key: {exc}")
                stop_event.set()
                return

            except (APIError, Exception) as exc:
                retries += 1
                wait = 2 ** retries
                logger.warning(f"QS stream error ({exc!r}) — retry {retries}/{max_retries} in {wait}s")
                time.sleep(wait)

        if retries >= max_retries:
            logger.critical("Max retries reached — signal stream stopped")
            stop_event.set()

    finally:
        signal_queue.put_nowait(None)

In [9]:
stop_event = threading.Event()
signal_queue: queue.Queue = queue.Queue()

stream_thread = threading.Thread(
    target=_signal_producer,
    args=(settings, stop_event, signal_queue),
    name="qs-stream",
    daemon=True,
)
stream_thread.start()

logger.info("Signal stream thread started (daemon=True)")
logger.info("Signals will arrive as SSE events and be placed on the queue")

19:25:26 | INFO     | Signal stream thread started (daemon=True)
19:25:26 | INFO     | Signals will arrive as SSE events and be placed on the queue


2026-05-06 19:25:26 [info     ] client_initialized             base_url=https://api.quantumsignals.ai/q version=0.4.0


19:25:26 | SUCCESS  | QuantumSignals SSE stream connected


---

## Step 5 — Resolve & Inspect IB Contracts

Before placing any order, IB needs a fully qualified `Contract` object. `gw.resolve_contract(symbol)` looks up the symbol in `settings.symbol_map`, constructs an `ib_async.Contract`, then calls `ib.qualifyContracts()` to fill in the `conId` (IB's unique contract identifier). Results are cached.

Default symbol map supports:
- **Futures:** ES, NQ, YM, RTY, CL, GC, MES, MNQ (CME/CBOT/NYMEX/COMEX)
- **Equities:** AAPL, SPY, QQQ (SMART exchange)

In [10]:
resolved_contracts = {}
symbols_to_check = list(settings.allowed_symbols) or ["ES", "NQ"]

for sym in symbols_to_check:
    logger.info(f"Resolving {sym}…")
    contract = gw.resolve_contract(sym)
    if contract is not None:
        resolved_contracts[sym] = contract
        logger.success(f"  {sym} → {contract}")
    else:
        logger.warning(f"  {sym} → could not qualify (check expiry month in config.py)")

logger.info(f"\nResolved {len(resolved_contracts)}/{len(symbols_to_check)} contracts")

19:25:33 | INFO     | Resolving ES…
19:25:33 | INFO     | IB qualifyContracts for ES (month=202606)…
19:25:34 | INFO     | IB qualifyContracts for ES finished in 0.76s (1 leg(s))
19:25:34 | SUCCESS  |   ES → Contract(secType='FUT', conId=649180678, symbol='ES', lastTradeDateOrContractMonth='20260618', multiplier='50', exchange='CME', currency='USD', localSymbol='ESM6', tradingClass='ES')
19:25:34 | INFO     | Resolving NQ…
19:25:34 | INFO     | IB qualifyContracts for NQ (month=202606)…
19:25:35 | INFO     | IB qualifyContracts for NQ finished in 0.50s (1 leg(s))
19:25:35 | SUCCESS  |   NQ → Contract(secType='FUT', conId=750150196, symbol='NQ', lastTradeDateOrContractMonth='20260618', multiplier='20', exchange='CME', currency='USD', localSymbol='NQM6', tradingClass='NQ')
19:25:35 | INFO     | 
Resolved 2/2 contracts


---

## Step 6 — Inspect Open Positions

`gw.get_positions_for_sync()` returns a list of dicts: `{"symbol": str, "qty": int, "avg_price": float}`.

Strategies use this at startup to:
- **Momentum:** re-hydrate in-memory position state
- **SignalFollower:** close any leftover positions from a previous session

In [11]:
positions = gw.get_positions_for_sync()

if not positions:
    logger.info("No open positions — clean slate, ready to trade")
else:
    logger.info(f"Found {len(positions)} open position(s):")
    for p in positions:
        label = "LONG " if p["qty"] > 0 else "SHORT"
        logger.info(f"  {label} {abs(p['qty']):>4d}x {p['symbol']:<8s} avg @ {p['avg_price']:.2f}")

19:25:42 | INFO     | No open positions — clean slate, ready to trade


---

## Step 7 — Follow Signals: The Main Event Loop

This is the heart of the bot. The loop runs on the main thread (required by `ib_async`) and does three things on every iteration:

1. **Drain the signal queue** — consume every signal the background thread has pushed, and act on each one.
2. **Tick the strategy** every second — the `signal_follower` strategy uses `tick()` to detect the 3:55–4:00 PM close window even when signals stop arriving.
3. **Print a status report** every 60 seconds.

### Signal handling (simplified)

| Signal | Action |
|---|---|
| `UP` | BUY 1 share of the mapped ETF (ES→SPY, NQ→QQQ) |
| `DOWN` | SELL 1 share of the mapped ETF |
| `STABLE` / null | No order placed |

Orders are placed via `gw.place_order(contract, action, qty)` — by default a market order, polled up to 5 seconds for a fill.

In [12]:
SIGNAL_TO_TRADE = {"ES": "SPY", "NQ": "QQQ"}
SIG_LABEL = {0: "DOWN  ↓", 1: "STABLE →", 2: "UP    ↑"}


def handle_signal(signal, gw, resolved_contracts, settings):
    """Act on a single QS signal: resolve contract and place order if actionable."""
    sym = signal.symbol

    if signal.is_null_signal():
        logger.debug(f"  [null] {sym}  model={signal.model}  reason={signal.code_message}")
        return False

    if settings.allowed_symbols and sym not in settings.allowed_symbols:
        return False

    sig_val = signal.signal
    label = SIG_LABEL.get(sig_val, f"?({sig_val})")
    logger.info(f"Signal  {sym:<4s}  {label}  model={signal.model}  ts={signal.for_minute_utc}")

    if sig_val not in (Signal.UP, Signal.DOWN):
        return False

    trade_sym = SIGNAL_TO_TRADE.get(sym, sym)
    contract = resolved_contracts.get(trade_sym)
    if contract is None:
        contract = gw.resolve_contract(trade_sym)
        if contract is None:
            logger.warning(f"  No contract for {trade_sym} — skipping")
            return False
        resolved_contracts[trade_sym] = contract

    action = "BUY" if sig_val == Signal.UP else "SELL"
    qty = settings.position_sizes.get(trade_sym, 1)

    logger.info(f"  → Placing {action} {qty}x {trade_sym}")
    trade = gw.place_order(contract, action, qty)

    if trade is not None:
        if isinstance(trade, dict):
            fill = trade.get("price")
        else:
            fill = getattr(getattr(trade, "orderStatus", None), "avgFillPrice", None)
        fill_str = f"@ {fill:.2f}" if fill else "(price pending)"
        logger.success(f"  → Order complete: {action} {qty}x {trade_sym} {fill_str}")

    return True

Configure the loop parameters and run. In tutorial mode the loop processes up to **5 actionable signals** then stops. Set `MAX_SIGNALS = 0` below to run indefinitely.

In [13]:
MAX_SIGNALS = 5  # set to 0 to run indefinitely until Ctrl-C

perf = PerformanceTracker()
signals_acted_on = 0
poll_s = 0.05
report_interval_s = 60.0
next_report_at = time.monotonic() + report_interval_s

if MAX_SIGNALS > 0:
    logger.info(f"Tutorial mode: will process up to {MAX_SIGNALS} actionable signals then stop")
logger.info("Press Ctrl-C (or interrupt kernel) at any time to stop")

try:
    while not stop_event.is_set():
        # A. Drain the signal queue
        try:
            while True:
                item = signal_queue.get_nowait()
                if item is None:
                    logger.warning("Signal stream ended — stopping loop")
                    stop_event.set()
                    break
                try:
                    if handle_signal(item, gw, resolved_contracts, settings):
                        signals_acted_on += 1
                except Exception as exc:
                    logger.exception(f"Error handling signal: {exc}")
        except queue.Empty:
            pass

        # Tutorial exit condition
        if MAX_SIGNALS > 0 and signals_acted_on >= MAX_SIGNALS:
            logger.success(f"Processed {signals_acted_on} actionable signals — tutorial complete")
            break

        # B. Periodic status report
        now = time.monotonic()
        if now >= next_report_at:
            logger.info("─" * 50)
            logger.info("STATUS REPORT")
            try:
                acct = gw.get_account_summary()
                logger.info(f"  Net liquidation : ${acct['net_liquidation']:>12,.2f}")
                logger.info(f"  Unrealised PnL  : ${acct['unrealized_pnl']:>12,.2f}")
            except Exception:
                pass
            if perf.total_trades:
                logger.info(f"  {perf.summary()}")
            logger.info("─" * 50)
            next_report_at = now + report_interval_s

        time.sleep(poll_s)

except KeyboardInterrupt:
    logger.info("Interrupted — stopping")
finally:
    stop_event.set()

19:25:56 | INFO     | Tutorial mode: will process up to 5 actionable signals then stop
19:25:56 | INFO     | Press Ctrl-C (or interrupt kernel) at any time to stop
19:26:15 | INFO     | Signal  NQ    UP    ↑  model=Pythia-aB24X  ts=2026-05-06 18:25:00+00:00
19:26:15 | INFO     | IB qualifyContracts for QQQ (month=n/a)…
19:26:16 | INFO     | IB qualifyContracts for QQQ finished in 1.26s (1 leg(s))
19:26:16 | INFO     |   → Placing BUY 5x QQQ
19:26:16 | INFO     | IB placeOrder BUY 5x QQQ…
19:26:16 | INFO     | Order placed  BUY 5x QQQ  MKT
19:26:17 | SUCCESS  |   → Order complete: BUY 5x QQQ @ 693.46
19:26:17 | INFO     | Signal  ES    UP    ↑  model=Pythia-aB24X  ts=2026-05-06 18:25:00+00:00
19:26:17 | INFO     | IB qualifyContracts for SPY (month=n/a)…
19:26:18 | INFO     | IB qualifyContracts for SPY finished in 0.67s (1 leg(s))
19:26:18 | INFO     |   → Placing BUY 5x SPY
19:26:18 | INFO     | IB placeOrder BUY 5x SPY…
19:26:18 | INFO     | Order placed  BUY 5x SPY  MKT
19:26:18 | S

---

## Step 8 — Close Positions & Disconnect

If `CLOSE_ON_EXIT=true` (default), the bot flattens all open positions before disconnecting so you never carry overnight exposure unintentionally.

In [14]:
if settings.close_on_exit:
    positions = gw.get_positions_for_sync()
    if positions:
        logger.info(f"Closing {len(positions)} open position(s) before disconnect…")
        for p in positions:
            sym, qty = p["symbol"], int(p["qty"])
            if qty == 0:
                continue
            contract = gw.resolve_contract(sym)
            if contract is None:
                logger.error(f"  Cannot resolve {sym} — manual close required")
                continue
            action = "SELL" if qty > 0 else "BUY"
            gw.place_order(contract, action, abs(qty))
            logger.info(f"  Closed {'LONG' if qty > 0 else 'SHORT'} {abs(qty)}x {sym}")
    else:
        logger.info("No open positions — nothing to close")
else:
    logger.info("CLOSE_ON_EXIT=false — leaving positions open")

19:26:47 | INFO     | Closing 2 open position(s) before disconnect…
19:26:47 | INFO     | IB placeOrder SELL 5x QQQ…
19:26:47 | INFO     | Order placed  SELL 5x QQQ  MKT
19:26:48 | INFO     |   Closed LONG 5x QQQ
19:26:48 | INFO     | IB placeOrder SELL 5x SPY…
19:26:48 | INFO     | Order placed  SELL 5x SPY  MKT
19:26:48 | INFO     |   Closed LONG 5x SPY


### Session Summary

In [15]:
cap = settings.daily_start_capital

logger.info("═" * 50)
logger.info("SESSION SUMMARY")
logger.info(f"  Total PnL  : ${perf.gross_pnl:+,.2f}")
logger.info(f"  Return on ${cap:,.0f}: {perf.gross_pnl / cap:+.2%}")
if perf.total_trades:
    logger.info(f"  {perf.summary()}")
logger.info("═" * 50)

19:26:58 | INFO     | ══════════════════════════════════════════════════
19:26:58 | INFO     | SESSION SUMMARY
19:26:58 | INFO     |   Total PnL  : $+0.00
19:26:58 | INFO     |   Return on $100,000: +0.00%
19:26:58 | INFO     | ══════════════════════════════════════════════════


In [ ]:
gw.disconnect()
logger.success("Disconnected from IB Gateway — goodbye!")